LSTM-based model for Tagalog NSFW word detection

In [1]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, average_precision_score, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import subprocess
import sys
import logging

class Logger:
    def __init__(self):
        self.logger = logging.getLogger("NSFW_Model")
        self.logger.setLevel(logging.INFO)
        # Avoid duplicate handlers in Colab
        if not self.logger.handlers:
            handler = logging.StreamHandler(sys.stdout)
            handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s', datefmt='%H:%M:%S'))
            self.logger.addHandler(handler)

    def set_log_file(self, path):
        handler = logging.FileHandler(path)
        handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
        self.logger.addHandler(handler)
        self.info(f"Logging to file: {path}")

    def info(self, message):
        self.logger.info(message)

    def warning(self, message):
        self.logger.warning(message)

    def error(self, message):
        self.logger.error(message)

    def exception(self, message):
        self.logger.exception(message)

    def section(self, title):
        self.info("\n" + "="*70)
        self.info(title.upper())
        self.info("="*70)

logger = Logger()

Check Hardware

In [2]:
class Config:
    def __init__(self):
        # Metadata
        self.model_version = "1.0.0"

        # Model Hyperparameters
        self.vocab_size = 10000
        self.max_sequence_length = 64
        self.embedding_dim = 64
        self.lstm_units = 64
        self.dense_units = 64
        self.dropout_rate = 0.5
        self.recurrent_dropout = 0.0  # 0 keeps the fast cuDNN LSTM kernel
        self.learning_rate = 0.001

        # Training Hyperparameters
        self.batch_size = 64
        self.epochs = 20

        # Dataset Split (70/15/15)
        self.validation_split = 0.15
        self.test_split = 0.15

        self.log_config()

    def log_config(self):
        """Print current configuration"""
        logger.section("MODEL CONFIGURATION")
        logger.info(f"Model Version: {self.model_version}")
        logger.info(f"Batch Size: {self.batch_size}")
        logger.info(f"Embedding Dimension: {self.embedding_dim}")
        logger.info(f"LSTM Units: {self.lstm_units}")
        logger.info(f"Dense Units: {self.dense_units}")
        logger.info(f"Max Sequence Length: {self.max_sequence_length}")
        logger.info(f"Vocabulary Size: {self.vocab_size}")
        logger.info(f"Epochs: {self.epochs}")
        logger.info(f"Dropout Rate: {self.dropout_rate}")
        logger.info(f"Learning Rate: {self.learning_rate}")
        logger.info(f"Validation Split: {self.validation_split}")
        logger.info(f"Test Split: {self.test_split}")

In [3]:
def install_requirements():
    required_packages = [
        'tensorflow',
        'pandas',
        'scikit-learn',
        'matplotlib',
        'seaborn',
        'unidecode',
        'emoji'
    ]

    for package in required_packages:
        try:
            # Handle packages where import name differs from package name
            import_name = package
            if package == 'scikit-learn':
                import_name = 'sklearn'

            __import__(import_name.split('[')[0])
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [4]:
print("Checking and installing dependencies...")
install_requirements()
print("All dependencies loaded successfully!")

Checking and installing dependencies...
Installing unidecode...
Installing emoji...
All dependencies loaded successfully!


In [5]:
def load_dataset(csv_path='cleaned_data.csv'):
    logger.info(f"Loading dataset from {csv_path}...")

    # Attempt to read CSV directly. Will raise FileNotFoundError if missing.
    df = pd.read_csv(csv_path)
    logger.info(f"Dataset loaded successfully. Shape: {df.shape}")

    # Validate required columns
    required_cols = ['text', 'label']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Clean and validate data
    df = df.dropna(subset=['text', 'label'])
    df['text'] = df['text'].astype(str).str.strip()

    # Map text labels to integers if needed (safe->0, nsfw->1)
    if df['label'].dtype == 'object':
        logger.info("Mapping text labels (safe/nsfw) to binary integers...")
        label_map = {'safe': 0, 'nsfw': 1}
        # Normalize: lowercase and strip whitespace
        df['label'] = df['label'].astype(str).str.lower().str.strip().map(label_map)

        # Handle unmapped labels
        if df['label'].isnull().any():
            logger.warning(f"Dropping {df['label'].isnull().sum()} rows with unknown labels")
            df = df.dropna(subset=['label'])

    df['label'] = df['label'].astype(int)

    # Remove empty texts
    df = df[df['text'].str.len() > 0]

    # Validate labels are binary
    unique_labels = df['label'].unique()
    if not all(label in [0, 1] for label in unique_labels):
        raise ValueError("Labels must be 0 (safe) or 1 (nsfw)")

    logger.info(f"Dataset validation complete. Clean shape: {df.shape}")
    logger.info(f"Label distribution:\n{df['label'].value_counts()}")

    return df

In [6]:
def preprocess_data(df, config):
    from unidecode import unidecode
    import emoji
    import random

    logger.info("Preprocessing data with Augmentation...")

    def normalize_text(text):
        """Normalize text: demojize, leetspeak, noise removal, etc."""
        if not isinstance(text, str):
            return str(text)

        # 1. Emojification (Demojize)
        text = emoji.demojize(text, delimiters=(" ", " "))

        # 2. Leetspeak / Obfuscation Conversion
        # Common substitutions in Tagalog/English NSFW context
        leetspeak_map = {
            '@': 'a', '4': 'a',
            '3': 'e',
            '1': 'i', '!': 'i',
            '0': 'o',
            '5': 's', '$': 's',
            '7': 't', '+': 't'
        }
        for char, replacement in leetspeak_map.items():
            text = text.replace(char, replacement)

        # 2.5 Taglish Handling: Separate Tagalog prefixes from English/Root words
        prefixes = r'(mag|nag|pag|man|pan|ma|na|ka|pa|ipa|ika|maka|naka|i|ipag|ig)'
        text = re.sub(r'\b' + prefixes + r'-([a-zA-Z]{2,})\b', r'\1 \2', text, flags=re.IGNORECASE)

        # 3. Remove inserted noise (e.g. "f-u-c-k" -> "fuck")
        text = re.sub(r'(?<=[a-zA-Z])[^a-zA-Z0-9\s](?=[a-zA-Z])', '', text)

        # 4. Remove Diacritics
        text = unidecode(text)

        # 5. Reduce repeated characters
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)

        return text

    def augment_text(text):
        """Apply random character-level noise to simulate typos/obfuscation"""
        return text  # augmentation disabled: dataset is near-balanced
        if not isinstance(text, str) or len(text) < 5:
            return text

        # 50% chance to return original (handled by loop logic, but check here too)
        if random.random() > 0.5:
            return text

        # Choose augmentation type
        rand = random.random()
        if rand < 0.33:
             # Repetition (e.g., "bobo" -> "boobo")
            idx = random.randint(0, len(text)-1)
            return text[:idx] + text[idx]*2 + text[idx+1:]
        elif rand < 0.66:
            # Deletion (e.g., "bobo" -> "boo")
            idx = random.randint(0, len(text)-1)
            return text[:idx] + text[idx+1:]
        else:
            # Swap (e.g., "bobo" -> "obbo")
            idx = random.randint(0, len(text)-2)
            return text[:idx] + text[idx+1] + text[idx] + text[idx+2:]

    # Split data FIRST (on raw text) to avoid leakage and allow raw augmentation
    X = df['text'].values
    y = df['label'].values

    # 1st Split: Train vs (Val + Test)
    # Remaining portion size = validation_split + test_split
    rest_size = config.validation_split + config.test_split

    X_train_raw, X_rest_raw, y_train, y_rest = train_test_split(
        X, y, test_size=rest_size,
        random_state=42, stratify=y
    )

    # 2nd Split: Val vs Test
    # Calculate relative test size from the remaining portion
    # e.g., if rest is 0.3 and test is 0.05, relative size is 0.05/0.3 = 1/6
    if rest_size > 0:
        test_size_relative = config.test_split / rest_size
    else:
        test_size_relative = 0

    X_val_raw, X_test_raw, y_val, y_test = train_test_split(
        X_rest_raw, y_rest, test_size=test_size_relative,
        random_state=42, stratify=y_rest
    )

    logger.info(f"Training samples: {len(X_train_raw)}")
    logger.info(f"Validation samples: {len(X_val_raw)}")
    logger.info(f"Test samples: {len(X_test_raw)}")

    # Apply Augmentation to Training Data
    logger.info("Augmentation disabled (dataset is near-balanced)...")
    X_train_aug = []
    y_train_aug = []

    for text, label in zip(X_train_raw, y_train):
        # Always keep original
        X_train_aug.append(text)
        y_train_aug.append(label)

        # Generate augmented version
        # We augment ALL classes to improve general robustness against typos
        aug_text = augment_text(str(text))
        if aug_text != text:
            X_train_aug.append(aug_text)
            y_train_aug.append(label)

    X_train_raw = np.array(X_train_aug)
    y_train = np.array(y_train_aug)

    logger.info(f"Augmented Training samples: {len(X_train_raw)}")

    # Apply Normalization to all sets
    logger.info("Normalizing text...")
    X_train = [normalize_text(t) for t in X_train_raw]
    X_val = [normalize_text(t) for t in X_val_raw]
    X_test = [normalize_text(t) for t in X_test_raw]

    # Initialize tokenizer with aligned strategy for NSFW detection
    tokenizer = Tokenizer(
        num_words=config.vocab_size,
        oov_token='<OOV>',
        filters='\t\n',
        lower=False
    )

    # Fit tokenizer on training data
    tokenizer.fit_on_texts(X_train)

    # Convert texts to sequences
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)
    X_test_seq = tokenizer.texts_to_sequences(X_test)

    # Pad sequences
    X_train_pad = pad_sequences(
        X_train_seq,
        maxlen=config.max_sequence_length,
        padding='post',
        truncating='post'
    )
    X_val_pad = pad_sequences(
        X_val_seq,
        maxlen=config.max_sequence_length,
        padding='post',
        truncating='post'
    )
    X_test_pad = pad_sequences(
        X_test_seq,
        maxlen=config.max_sequence_length,
        padding='post',
        truncating='post'
    )

    logger.info(f"Vocabulary size: {len(tokenizer.word_index) + 1}")
    logger.info(f"Sequence shape: {X_train_pad.shape}")

    return X_train_pad, X_val_pad, X_test_pad, y_train, y_val, y_test, tokenizer, X_test_raw

# LSTM MODEL ARCHITECTURE

In [7]:
def build_lstm_model(config, vocab_size):
    logger.info("Building LSTM model with Focal Loss...")

    # Define Focal Loss
    def focal_loss(gamma=2.0, alpha=0.25):
        def focal_loss_fixed(y_true, y_pred):
            # Ensure y_true is float32 matches y_pred
            y_true = tf.cast(y_true, tf.float32)

            # Clip predictions to prevent NaN
            epsilon = tf.keras.backend.epsilon()
            y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

            # Calculate p_t
            p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)

            # Calculate alpha_t
            alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

            # Calculate Focal Loss
            fl = -alpha_t * tf.pow(1.0 - p_t, gamma) * tf.math.log(p_t)
            return tf.reduce_mean(fl)
        return focal_loss_fixed

    model = keras.Sequential([
        # Embedding layer
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=config.embedding_dim,
            mask_zero=True,  # ignore zero-padding so short texts aren't washed out
            name='embedding'
        ),

        # Spatial Dropout
        layers.SpatialDropout1D(0.2),

        # Single LSTM layer (reduced capacity to curb overfitting)
        layers.LSTM(
            config.lstm_units,
            dropout=config.dropout_rate,
            recurrent_dropout=config.recurrent_dropout
        ),

        # Dense head with L2 regularization
        layers.Dense(config.dense_units, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.Dropout(config.dropout_rate),

        # Output layer (binary classification)
        layers.Dense(1, activation='sigmoid')
    ])

    # Compile model
    optimizer = keras.optimizers.Adam(learning_rate=config.learning_rate)

    # Use Focal Loss
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
    )

    logger.info("\nModel Summary:")
    model.summary(print_fn=logger.info)

    return model

# MODEL TRAINING

In [8]:
def train_model(model, X_train, y_train, X_val, y_val, config, output_dir):
    from sklearn.utils.class_weight import compute_class_weight
    import pandas as pd

    logger.info("\nStarting model training...")

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Define callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(output_dir, 'best_model.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=config.batch_size,
        epochs=config.epochs,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    # Save final model
    model.save(os.path.join(output_dir, 'final_model.keras'))
    logger.info(f"\nModel saved to {output_dir}")

    # --- Save Training History to CSV ---
    history_df = pd.DataFrame(history.history)
    history_csv_path = os.path.join(output_dir, 'training_history.csv')
    history_df.to_csv(history_csv_path, index_label='epoch')
    logger.info(f"Training history saved to {history_csv_path}")
    # ------------------------------------

    return history

# MODEL EVALUATION AND VISUALIZATION

In [9]:
def evaluate_model(model, X_val, y_val, output_dir, X_test=None, y_test=None):
    """Evaluate model and generate metrics"""
    import pandas as pd
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report

    logger.info("\nEvaluating model...")

    # --- Threshold Tuning (Using Validation Set) ---
    # We always calculate the optimal threshold on the VALIDATION set to avoid data leakage
    if X_test is not None:
        # If running test evaluation, we assume threshold is already passed or we re-calculate it on Val first
        # Ideally, we should pass the threshold in, but for this pipeline, we'll re-calculate it on Val
        # to ensure consistency if called separately.
        pass

    logger.info("Tuning decision threshold on Validation Set...")
    y_val_prob = model.predict(X_val, verbose=0).flatten()

    # Search a sane band, and only deviate from the neutral 0.5 if the validation
    # gain is large enough to be real rather than noise (the old argmax over
    # 0.1-0.95 over-fit the threshold to the val split and hurt test results).
    thresholds = np.arange(0.30, 0.71, 0.01)
    f1_scores = [f1_score(y_val, (y_val_prob > t).astype(int)) for t in thresholds]
    best_idx = int(np.argmax(f1_scores))
    tuned_threshold = float(thresholds[best_idx])
    tuned_f1 = f1_scores[best_idx]

    f1_at_half = f1_score(y_val, (y_val_prob > 0.5).astype(int))
    if tuned_f1 - f1_at_half > 0.01:
        best_threshold, best_nsfw_f1 = tuned_threshold, tuned_f1
    else:
        best_threshold, best_nsfw_f1 = 0.5, f1_at_half

    logger.info(f"Optimal Threshold: {best_threshold:.4f} (tuned {tuned_threshold:.2f}->F1 {tuned_f1:.4f} vs 0.50->F1 {f1_at_half:.4f})")
    logger.info(f"Validation NSFW F1: {best_nsfw_f1:.4f}")

    # --- Final Evaluation (Using Test Set if available, else Validation) ---
    if X_test is not None and y_test is not None:
        logger.info("Calculating final metrics on Test Set using optimal threshold...")
        X_eval = X_test
        y_eval = y_test
        eval_name = "Test"
    else:
        logger.info("Calculating final metrics on Validation Set...")
        X_eval = X_val
        y_eval = y_val
        eval_name = "Validation"

    y_pred_prob = model.predict(X_eval, verbose=0)
    y_pred_prob_flat = y_pred_prob.flatten()
    y_pred = (y_pred_prob > best_threshold).astype(int).flatten()

    # Calculate metrics
    accuracy = accuracy_score(y_eval, y_pred)
    roc_auc = roc_auc_score(y_eval, y_pred_prob_flat)
    auprc = average_precision_score(y_eval, y_pred_prob_flat)

    # Weighted/Macro metrics
    macro_f1 = f1_score(y_eval, y_pred, average='macro')
    macro_precision = precision_score(y_eval, y_pred, average='macro')
    macro_recall = recall_score(y_eval, y_pred, average='macro')

    nsfw_f1 = f1_score(y_eval, y_pred) # Binary default is pos_label=1

    report = classification_report(
        y_eval, y_pred,
        target_names=['Safe', 'NSFW'],
        digits=4
    )

    # Confusion matrix
    cm = confusion_matrix(y_eval, y_pred)

    # Save metrics text
    metrics_path = os.path.join(output_dir, f'{eval_name.lower()}_evaluation_metrics.txt')
    metrics_text = f"""LSTM Tagalog NSFW Detection Model Evaluation
{'='*70}

Model Architecture: LSTM
Task: Binary Classification (Safe vs NSFW)
Evaluation Set: {eval_name}

Calibration:
----------------------------------------------------------------------
Optimized Threshold: {best_threshold:.4f} (Tuned on Validation)
Target Metric:      Max F1 (NSFW)
----------------------------------------------------------------------

Key Metrics ({eval_name}):
----------------------------------------------------------------------
Overall Accuracy:   {accuracy:.4f}
Macro Precision:    {macro_precision:.4f}
Macro Recall:       {macro_recall:.4f}
Macro F1-Score:     {macro_f1:.4f}
NSFW F1-Score:      {nsfw_f1:.4f}
AUROC:              {roc_auc:.4f}
PR-AUC (AUPRC):     {auprc:.4f}
----------------------------------------------------------------------

Classification Report:
{report}

Confusion Matrix:
{cm}
"""

    with open(metrics_path, 'w') as f:
        f.write(metrics_text)

    # --- Save Metrics to CSV ---
    metrics_data = {
        'set': eval_name,
        'accuracy': accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'nsfw_f1': nsfw_f1,
        'roc_auc': roc_auc,
        'pr_auc': auprc,
        'optimal_threshold': best_threshold
    }
    metrics_df = pd.DataFrame([metrics_data])
    # Dynamic filename based on set (Validation vs Test)
    metrics_csv_path = os.path.join(output_dir, f'{eval_name.lower()}_metrics.csv')
    metrics_df.to_csv(metrics_csv_path, index=False)
    logger.info(f"{eval_name} metrics saved to {metrics_csv_path}")

    # Save detailed classification report to CSV
    report_dict = classification_report(
        y_eval, y_pred,
        target_names=['Safe', 'NSFW'],
        output_dict=True
    )
    report_df = pd.DataFrame(report_dict).transpose()
    report_csv_path = os.path.join(output_dir, f'{eval_name.lower()}_classification_report.csv')
    report_df.to_csv(report_csv_path)
    logger.info(f"{eval_name} classification report saved to {report_csv_path}")

    # --- Save Detailed Predictions to CSV ---
    predictions_df = pd.DataFrame({
        'y_true': y_eval,
        'y_pred': y_pred,
        'y_prob': y_pred_prob_flat
    })
    predictions_csv_path = os.path.join(output_dir, f'{eval_name.lower()}_predictions.csv')
    predictions_df.to_csv(predictions_csv_path, index=False)
    logger.info(f"{eval_name} predictions saved to {predictions_csv_path}")
    # ---------------------------

    logger.info(f"{eval_name} Accuracy: {accuracy:.4f}")
    logger.info(f"{eval_name} Macro F1-Score: {macro_f1:.4f}")
    logger.info(f"Metrics saved to {metrics_path}")

    # Plot confusion matrix
    plot_confusion_matrix(cm, output_dir, filename=f'{eval_name.lower()}_confusion_matrix.png')

    return accuracy, report, best_threshold

In [10]:
def plot_confusion_matrix(cm, output_dir, filename='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Safe', 'NSFW'],
        yticklabels=['Safe', 'NSFW']
    )
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path)
    logger.info(f"Confusion matrix saved to {save_path}")
    plt.close()

In [11]:
def plot_training_history(history, output_dir):
    """Plot training history"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # Loss plot
    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'))
    logger.info(f"Training history saved to {output_dir}/training_history.png")
    plt.close()

# MODEL EXPORT

In [12]:
def save_artifacts(model, tokenizer, config, output_dir, threshold=0.5):
    """Save model artifacts for deployment"""
    logger.info("\nSaving model artifacts...")

    # Save tokenizer
    tokenizer_path = os.path.join(output_dir, 'tokenizer.pickle')
    with open(tokenizer_path, 'wb') as f:
        pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)
    logger.info(f"Tokenizer saved to {tokenizer_path}")

    # Save config
    config_dict = {
        'model_version': config.model_version,
        'decision_threshold': float(threshold),
        'max_sequence_length': config.max_sequence_length,
        'vocab_size': config.vocab_size,
        'embedding_dim': config.embedding_dim,
        'lstm_units': config.lstm_units,
        'dense_units': config.dense_units
    }

    config_path = os.path.join(output_dir, 'config.json')
    with open(config_path, 'w') as f:
        json.dump(config_dict, f, indent=4)
    logger.info(f"Configuration saved to {config_path}")

    # Export to TFLite
    export_to_tflite(model, config, output_dir)

In [13]:
def export_to_tflite(model, config, output_dir):
    logger.info("\nExporting to TensorFlow Lite...")

    try:
        # Convert to TFLite
        # The cuDNN LSTM kernel (recurrent_dropout=0 on GPU) emits a 'CudnnRNNV3'
        # op that TFLite cannot convert (not a builtin and not a flex/Select-TF op).
        # Rebuild an inference twin on the generic kernel (use_cudnn=False) and copy
        # the trained weights; weight layout is identical so predictions are unchanged.
        def _strip_cudnn(layer):
            cfg = layer.get_config()
            if isinstance(layer, layers.LSTM):
                cfg['use_cudnn'] = False
                return layers.LSTM.from_config(cfg)
            if isinstance(layer, layers.Bidirectional):
                inner = cfg.get('layer', {}).get('config')
                if inner is not None:
                    inner['use_cudnn'] = False
                return layers.Bidirectional.from_config(cfg)
            return layer.__class__.from_config(cfg)

        inference_model = keras.models.clone_model(model, clone_function=_strip_cudnn)
        inference_model.build(model.input_shape)
        inference_model.set_weights(model.get_weights())

        converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)

        # Optimization settings
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

        # FIX: Enable Select TF Ops and disable TensorList lowering
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS
        ]
        converter._experimental_lower_tensor_list_ops = False

        # Convert model
        tflite_model = converter.convert()

        # Save TFLite model
        tflite_path = os.path.join(output_dir, 'model.tflite')
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)

        model_size = os.path.getsize(tflite_path) / (1024 * 1024)
        logger.info(f"TFLite model saved to {tflite_path}")
        logger.info(f"TFLite model size: {model_size:.2f} MB")

        return True
    except Exception as e:
        logger.error(f"TFLite export failed: {e}")
        return False

# INFERENCE

In [14]:
def test_inference(model_path, tokenizer_path, config_path, test_texts=None, thresholds=None):
    """Test model inference with sample texts using multiple thresholds"""
    from unidecode import unidecode
    import emoji
    import re
    import json
    import pickle
    import numpy as np
    import pandas as pd
    import os
    from tensorflow import keras
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    if test_texts is None:
        test_texts = [
            "Magandang umaga sa lahat",
            "Kumusta ka",
            "Salamat po",
            "Good morning everyone",
            "inappropriate example",
            # Obfuscated / Stylized tests
            "p@ke",
            "F-U-C-K",
            "sh!t",
            "b4d content",
            "I am sooooo angry 😡", # Normalization test
            # Taglish tests
            "Nag-enjoy ako",
            "Mag-drive ka na",
            "Naka-lock yung pinto"
        ]

    logger.info("\nTesting model inference...")

    # Determine output directory
    output_dir = os.path.dirname(model_path)

    def normalize_text(text):
        """Match training normalization"""
        if not isinstance(text, str):
            return str(text)

        text = emoji.demojize(text, delimiters=(" ", " "))

        # Leetspeak
        leetspeak_map = {
            '@': 'a', '4': 'a',
            '3': 'e',
            '1': 'i', '!': 'i',
            '0': 'o',
            '5': 's', '$': 's',
            '7': 't', '+': 't'
        }
        for char, replacement in leetspeak_map.items():
            text = text.replace(char, replacement)

        # Taglish Handling
        prefixes = r'(mag|nag|pag|man|pan|ma|na|ka|pa|ipa|ika|maka|naka|i|ipag|ig)'
        text = re.sub(r'\b' + prefixes + r'-([a-zA-Z]{2,})\b', r'\1 \2', text, flags=re.IGNORECASE)

        # Noise removal
        text = re.sub(r'(?<=[a-zA-Z])[^a-zA-Z0-9\s](?=[a-zA-Z])', '', text)

        text = unidecode(text)
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)
        return text

    try:
        # Load model and tokenizer
        # NOTE: compile=False is used because we have a custom loss function (Focal Loss)
        # which might cause issues during loading if not registered.
        # Since we only do inference here, compilation is not strictly required.
        model = keras.models.load_model(model_path, compile=False)

        with open(tokenizer_path, 'rb') as f:
            tokenizer = pickle.load(f)

        with open(config_path, 'r') as f:
            config = json.load(f)

        # Setup thresholds
        if thresholds is None:
            calibrated = config.get('decision_threshold', 0.5)
            thresholds = {
                'Calibrated': calibrated,
                'Lenient': 0.2,
                'Standard': 0.5,
                'Strict': 0.8
            }

        logger.info("Inference Configuration:")
        for name, val in thresholds.items():
            logger.info(f"  {name} Threshold: {val:.4f}")

        results_data = []

        # Process and predict
        for text in test_texts:
            # Apply normalization
            norm_text = normalize_text(text)

            seq = tokenizer.texts_to_sequences([norm_text])
            padded = pad_sequences(
                seq,
                maxlen=config['max_sequence_length'],
                padding='post'
            )

            pred_prob = model.predict(padded, verbose=0)[0][0]

            logger.info("-" * 40)
            logger.info(f"Text: '{text}'")
            if text != norm_text:
                logger.info(f"Norm: '{norm_text}'")
            logger.info(f"Confidence Score: {pred_prob:.4f}")

            # Store Result Data
            row = {
                'text': text,
                'normalized_text': norm_text,
                'confidence_score': float(pred_prob)
            }

            # Check against all thresholds
            for name, threshold in thresholds.items():
                pred_class = 1 if pred_prob > threshold else 0
                label = "NSFW" if pred_class == 1 else "Safe"
                logger.info(f"  [{name} @ {threshold:.2f}] -> {label}")

                row[f'{name}_label'] = label
                row[f'{name}_threshold'] = threshold

            results_data.append(row)

        # Save results to CSV
        if results_data:
            results_df = pd.DataFrame(results_data)
            results_csv_path = os.path.join(output_dir, 'inference_results.csv')
            results_df.to_csv(results_csv_path, index=False)
            logger.info(f"\nInference results saved to {results_csv_path}")

    except Exception as e:
        logger.error(f"Inference test failed: {e}")
        import traceback
        traceback.print_exc()

In [15]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os
import pandas as pd

# ============================================
# Model Trainer Class (Fixed Hyperparameters, No Tuner)
# ============================================

class ModelTrainer:
    def __init__(self, output_dir="models/lstm_tagalog_nsfw"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        logger.set_log_file(os.path.join(output_dir, 'training.log'))

        self.config = Config()
        self.tokenizer = None
        self.vocab_size = 0
        self.model = None
        self.history = None
        self.X_train, self.X_val, self.X_test = None, None, None
        self.y_train, self.y_val, self.y_test = None, None, None
        self.X_test_raw = None
        self.best_threshold = 0.5

    def prepare_data(self):
        """Step 1: Load and Preprocess Data"""
        logger.section("STEP 1: DATA PREPARATION")
        df = load_dataset()
        self.X_train, self.X_val, self.X_test, self.y_train, self.y_val, self.y_test, self.tokenizer, self.X_test_raw = preprocess_data(df, self.config)
        self.vocab_size = min(len(self.tokenizer.word_index) + 1, self.config.vocab_size)
        logger.info("Data preparation complete.")

    def build_model(self):
        """Step 2: Build model with fixed hyperparameters (no tuning)"""
        logger.section("STEP 2: MODEL BUILDING")
        if self.X_train is None:
            raise ValueError("Data not loaded. Run prepare_data() first.")

        self.model = build_lstm_model(self.config, self.vocab_size)

    def train(self):
        """Step 3: Train Model"""
        logger.section("STEP 3: TRAINING")
        if self.model is None:
            raise ValueError("Model not built. Run build_model() first.")

        # Training uses Train/Val
        self.history = train_model(
            self.model, self.X_train, self.y_train, self.X_val, self.y_val,
            self.config, self.output_dir
        )

    def evaluate(self):
        """Step 4: Evaluate Model on Validation Set"""
        logger.section("STEP 4: EVALUATION (VALIDATION SET)")
        if self.model is None:
            raise ValueError("Model not available for evaluation.")

        self.accuracy, self.report, self.best_threshold = evaluate_model(
            self.model, self.X_val, self.y_val, self.output_dir
        )
        logger.info(f"Final validation accuracy: {self.accuracy:.4f}")

    def test(self):
        """Step 5: Evaluate on Test Set (Unseen Data)"""
        logger.section("STEP 5: TEST SET EVALUATION")
        if self.X_test is None:
             raise ValueError("Test data not loaded.")
        if self.model is None:
             raise ValueError("Model not available.")

        test_acc, test_report, _ = evaluate_model(
            self.model, self.X_val, self.y_val, self.output_dir,
            X_test=self.X_test, y_test=self.y_test
        )
        logger.info(f"Final Test Set Accuracy: {test_acc:.4f}")

    def export(self):
        """Step 6: Export Artifacts"""
        logger.section("STEP 6: EXPORT")
        if self.history:
            plot_training_history(self.history, self.output_dir)

        save_artifacts(self.model, self.tokenizer, self.config, self.output_dir, self.best_threshold)
        logger.info(f"Pipeline complete. Artifacts in {self.output_dir}")
        return self.output_dir

    def run_all(self):
        """Run the full pipeline sequentially"""
        self.prepare_data()
        self.build_model()
        self.train()
        self.evaluate()
        self.test() # Explicit Test Step
        return self.export()

In [16]:
if __name__ == "__main__":
    try:
        # Initialize Trainer
        trainer = ModelTrainer()

        # Execute Pipeline Steps
        trainer.prepare_data()
        trainer.build_model()
        trainer.train()
        trainer.evaluate()
        trainer.test() # Evaluate on held-out test set
        output_dir = trainer.export()

        # Test inference with actual test data samples
        print("\nRunning inference test on Test Set samples...")
        import numpy as np

        if trainer.X_test_raw is not None and len(trainer.X_test_raw) > 0:
             sample_texts = trainer.X_test_raw[:20]
        else:
             print("Warning: No test data found. Using default samples.")
             sample_texts = None

        test_inference(
            model_path=os.path.join(output_dir, "final_model.keras"),
            tokenizer_path=os.path.join(output_dir, "tokenizer.pickle"),
            config_path=os.path.join(output_dir, "config.json"),
            test_texts=sample_texts
        )

    except KeyboardInterrupt:
        print("\nTraining interrupted by user.")
    except Exception as e:
        print(f"Error during execution: {e}")
        import traceback
        traceback.print_exc()

    print("\nProgram execution completed.")

07:37:11 - Logging to file: models/lstm_tagalog_nsfw/training.log


INFO:NSFW_Model:Logging to file: models/lstm_tagalog_nsfw/training.log


07:37:11 - 


INFO:NSFW_Model:


07:37:11 - MODEL CONFIGURATION


...

07:37:40 -   [Strict @ 0.80] -> Safe


INFO:NSFW_Model:  [Strict @ 0.80] -> Safe


07:37:40 - 
Inference results saved to models/lstm_tagalog_nsfw/inference_results.csv


INFO:NSFW_Model:
Inference results saved to models/lstm_tagalog_nsfw/inference_results.csv



Program execution completed.


In [17]:
import os
from google.colab import files

output_zip = 'LSTM.zip'
folders = ['models']

# Filter only existing folders
folders_to_zip = [f for f in folders if os.path.exists(f)]

if folders_to_zip:
    print(f"Zipping: {', '.join(folders_to_zip)}")
    cmd = f"zip -r {output_zip} {' '.join(folders_to_zip)}"
    os.system(cmd)

    if os.path.exists(output_zip):
        print(f"Downloading {output_zip} ({os.path.getsize(output_zip)/1e6:.2f} MB)...")
        files.download(output_zip)
    else:
        print("Error creating zip file.")
else:
    print("No folders found to zip.")

Zipping: models


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>